In [ ]:
"""
LLM-Only Baseline for Conversational Process Intelligence

Implementation of the LLM-only experimental condition described in:
"From Event Logs to Conversational Process Intelligence:
A Hybrid Architecture and Prototype for Natural Language-Based Process Analysis"

Processing flow:
    Raw event log
        -> XLSX-to-CSV text representation
        -> Input-integrity verification
        -> Stored base interaction containing the complete raw event log
        -> Independent natural-language questions
        -> LLM analysis and response

Experimental constraints:
    - No Process Mining metrics are computed in Python.
    - No analytical functions are available to the LLM.
    - No rule-based routing is performed.
    - No deterministic analytical results are provided to the LLM.
    - Every question independently references the same stored base interaction.
    - Question-answer pairs are not chained.

The system instruction, model, temperature, input representation, and experimental
logic below reproduce the configuration used for the LLM-only evaluation.
"""

# =============================================================================
# Installation and imports
# =============================================================================

!pip install -q google-genai openpyxl

import pandas as pd
import json
import time
import hashlib

from datetime import datetime
from getpass import getpass

from google import genai

import ipywidgets as widgets
from IPython.display import display, Markdown


# =============================================================================
# Configuration
# =============================================================================

EVENT_LOG_FILE = "EC 1 - Purchasing.xlsx"

RAW_CSV_FILE = "EC_1_Purchasing_raw.csv"

GEMINI_MODEL = "gemini-2.5-flash-lite"

TEMPERATURE = 0.0

RESULTS_FILE = "llm_only_final_results.csv"

CONFIGURATION_FILE = "llm_only_final_configuration.json"

MINIMUM_LLM_INTERVAL = 60


# =============================================================================
# Raw event-log representation
# =============================================================================
#
# The XLSX file is converted to CSV solely to obtain a
# text-compatible representation of the complete event log.
#
# No Process Mining metric, aggregation, classification,
# variant, rework group, case duration, waiting time,
# termination pattern, or other analytical result is
# computed here.

conversion_start = time.perf_counter()


raw_log = pd.read_excel(
    EVENT_LOG_FILE
)


original_rows = len(
    raw_log
)


original_columns = len(
    raw_log.columns
)


original_column_names = list(
    raw_log.columns
)


raw_log.to_csv(
    RAW_CSV_FILE,
    index=False,
    encoding="utf-8"
)


conversion_time = (
    time.perf_counter()
    -
    conversion_start
)


print("\n==========================================")
print("RAW EVENT LOG PREPARED")
print("==========================================")

print(
    "Source file:",
    EVENT_LOG_FILE
)

print(
    "Original XLSX rows:",
    original_rows
)

print(
    "Original XLSX columns:",
    original_columns
)

print(
    "Column names:",
    original_column_names
)

print(
    "Conversion time:",
    round(
        conversion_time,
        4
    ),
    "seconds"
)

print(
    "No process analytics were computed."
)

print("==========================================\n")


# =============================================================================
# Input-integrity verification
# =============================================================================
#
# This section verifies only whether the XLSX-to-CSV
# representation preserves the dataset dimensions and
# column structure.
#
# These checks are NOT Process Mining analyses and their
# results are NOT provided to the LLM.

verification_log = pd.read_csv(
    RAW_CSV_FILE
)


csv_rows = len(
    verification_log
)


csv_columns = len(
    verification_log.columns
)


csv_column_names = list(
    verification_log.columns
)


rows_preserved = (
    original_rows
    ==
    csv_rows
)


columns_preserved = (
    original_columns
    ==
    csv_columns
)


column_names_preserved = (
    original_column_names
    ==
    csv_column_names
)


integrity_passed = (
    rows_preserved
    and
    columns_preserved
    and
    column_names_preserved
)


if not integrity_passed:

    raise ValueError(
        "INPUT INTEGRITY VERIFICATION FAILED. "
        "The CSV representation does not preserve "
        "the XLSX row/column structure."
    )


print("==========================================")
print("INPUT INTEGRITY VERIFICATION")
print("==========================================")

print(
    "XLSX rows:",
    original_rows
)

print(
    "CSV rows:",
    csv_rows
)

print(
    "Rows preserved:",
    rows_preserved
)

print(
    "XLSX columns:",
    original_columns
)

print(
    "CSV columns:",
    csv_columns
)

print(
    "Columns preserved:",
    columns_preserved
)

print(
    "Column names preserved:",
    column_names_preserved
)

print(
    "INPUT INTEGRITY: PASSED"
)

print("==========================================\n")


# =============================================================================
# Load complete raw CSV as text
# =============================================================================

with open(
    RAW_CSV_FILE,
    "r",
    encoding="utf-8"
) as file:

    raw_log_text = file.read()


raw_csv_characters = len(
    raw_log_text
)


raw_csv_bytes = len(
    raw_log_text.encode("utf-8")
)


# =============================================================================
# SHA-256 fingerprint
# =============================================================================
#
# The SHA-256 hash identifies the exact textual
# representation used in the experiment.

raw_csv_sha256 = hashlib.sha256(
    raw_log_text.encode("utf-8")
).hexdigest()


print("==========================================")
print("RAW INPUT FINGERPRINT")
print("==========================================")

print(
    "Raw CSV characters:",
    raw_csv_characters
)

print(
    "Raw CSV bytes:",
    raw_csv_bytes
)

print(
    "Raw CSV SHA-256:",
    raw_csv_sha256
)

print("==========================================\n")


# =============================================================================
# System instruction
# =============================================================================

SYSTEM_INSTRUCTION = """
You are a Process Mining specialist.

You receive a complete raw event log and natural-language
questions about the process represented in that log.

Your task is to analyze the provided event log and answer
the user's question using only information that can be
supported by the event data.

Rules:

- Use only the provided event log as the source of process data.
- Treat the provided event-log text as the complete dataset
  available for the analysis.
- Do not invent numbers, facts, process characteristics, or
  relationships that cannot be supported by the event log.
- If the requested information cannot be determined from the
  available event log, state this explicitly.
- Do not use external knowledge to fill missing process data.
- Be technical, objective, and clear.
- Distinguish observed evidence from interpretation.
- Do not establish causal relationships when the available data
  only support association.
- Do not use the term "significantly" or equivalent expressions
  unless a statistical test has actually been performed and its
  result supports that statement.
- Do not assume that all cases are completed unless this can be
  established from the event log.
- Do not answer in JSON unless explicitly requested.
"""


# =============================================================================
# Gemini connection
# =============================================================================

api_key = getpass(
    "Enter your Google AI Studio API key: "
)


client = genai.Client(
    api_key=api_key
)


print(
    "Connected model:",
    GEMINI_MODEL
)

print(
    "Temperature:",
    TEMPERATURE
)


# =============================================================================
# Raw event-log token count
# =============================================================================
#
# This measures input size only.
# It does not calculate a process metric.

token_count_result = (
    client.models.count_tokens(
        model=GEMINI_MODEL,
        contents=raw_log_text
    )
)


raw_log_tokens = getattr(
    token_count_result,
    "total_tokens",
    None
)


print(
    "Raw event-log tokens:",
    raw_log_tokens
)


# =============================================================================
# Create base interaction
# =============================================================================
#
# The complete raw event log is sent once to create the
# experimental base interaction.
#
# Every subsequent experimental question independently
# references THIS SAME BASE.
#
# Q02 does NOT reference Q01.
# Q03 does NOT reference Q01 or Q02.
# Etc.

BASE_PROMPT = f"""
COMPLETE RAW EVENT LOG:

{raw_log_text}

The text above is the complete raw event log that will be
used as the data source for subsequent process-analysis
questions.

Do not perform process analysis in this interaction.

Reply only with:

EVENT LOG CONTEXT RECEIVED
"""


print("\nCreating experimental base interaction...")


base_start = time.perf_counter()


base_interaction = (
    client.interactions.create(

        model=GEMINI_MODEL,

        input=BASE_PROMPT,

        system_instruction=(
            SYSTEM_INSTRUCTION
        ),

        generation_config={
            "temperature":
                TEMPERATURE
        },

        store=True
    )
)


base_latency = (
    time.perf_counter()
    -
    base_start
)


BASE_INTERACTION_ID = (
    base_interaction.id
)


base_input_tokens = None
base_cached_tokens = None
base_output_tokens = None
base_total_tokens = None


try:

    base_usage = (
        base_interaction.usage
    )


    base_input_tokens = getattr(
        base_usage,
        "total_input_tokens",
        None
    )


    base_cached_tokens = getattr(
        base_usage,
        "total_cached_tokens",
        None
    )


    base_output_tokens = getattr(
        base_usage,
        "total_output_tokens",
        None
    )


    base_total_tokens = getattr(
        base_usage,
        "total_tokens",
        None
    )


except Exception:

    pass


print("\n==========================================")
print("BASE INTERACTION CREATED")
print("==========================================")

print(
    "Base interaction ID:",
    BASE_INTERACTION_ID
)

print(
    "Base response:",
    base_interaction.output_text
)

print(
    "Base latency:",
    round(
        base_latency,
        6
    ),
    "seconds"
)

print(
    "Base input tokens:",
    base_input_tokens
)

print(
    "Base cached tokens:",
    base_cached_tokens
)

print(
    "Base output tokens:",
    base_output_tokens
)

print(
    "Base total tokens:",
    base_total_tokens
)

print("==========================================\n")


# =============================================================================
# Reproducible configuration file
# =============================================================================

experiment_configuration = {

    "experiment_status":
        "FROZEN OFFICIAL LLM-ONLY BASELINE",

    "architecture":
        "LLM-only",

    "context_strategy":
        "server-side base interaction",

    "model":
        GEMINI_MODEL,

    "temperature":
        TEMPERATURE,

    "source_event_log":
        EVENT_LOG_FILE,

    "raw_csv_file":
        RAW_CSV_FILE,

    "input_representation":
        (
            "Complete raw event log converted from XLSX "
            "to CSV without analytical preprocessing."
        ),

    "xlsx_rows":
        original_rows,

    "xlsx_columns":
        original_columns,

    "column_names":
        original_column_names,

    "csv_rows":
        csv_rows,

    "csv_columns":
        csv_columns,

    "rows_preserved":
        rows_preserved,

    "columns_preserved":
        columns_preserved,

    "column_names_preserved":
        column_names_preserved,

    "input_integrity_passed":
        integrity_passed,

    "raw_csv_characters":
        raw_csv_characters,

    "raw_csv_bytes":
        raw_csv_bytes,

    "raw_csv_sha256":
        raw_csv_sha256,

    "raw_event_log_tokens":
        raw_log_tokens,

    "base_interaction_id":
        BASE_INTERACTION_ID,

    "base_input_tokens":
        base_input_tokens,

    "base_cached_tokens":
        base_cached_tokens,

    "base_output_tokens":
        base_output_tokens,

    "base_total_tokens":
        base_total_tokens,

    "base_latency_seconds":
        round(
            base_latency,
            6
        ),

    "question_independence":
        (
            "Every experimental question independently "
            "references the same raw-event-log base "
            "interaction. Question-answer pairs are "
            "not chained."
        ),

    "system_instruction":
        SYSTEM_INSTRUCTION
}


with open(
    CONFIGURATION_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        experiment_configuration,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    "Configuration saved to:",
    CONFIGURATION_FILE
)


# =============================================================================
# Experimental result storage
# =============================================================================

experiment_records = []


def save_records():

    if not experiment_records:

        return


    pd.DataFrame(
        experiment_records
    ).to_csv(
        RESULTS_FILE,
        index=False,
        encoding="utf-8-sig"
    )


# =============================================================================
# Experimental question function
# =============================================================================

def ask_about_log(
    question,
    question_id
):

    question = (
        str(question)
        .strip()
    )


    question_id = (
        str(question_id)
        .strip()
        .upper()
    )


    if question == "":

        raise ValueError(
            "Question cannot be empty."
        )


    if question_id == "":

        raise ValueError(
            "Question ID cannot be empty."
        )


    # Prevent accidental duplicate official IDs
    used_ids = {
        str(record["question_id"]).upper()
        for record in experiment_records
        if record.get("question_id") is not None
    }


    if question_id in used_ids:

        raise ValueError(
            f"Question ID {question_id} "
            "has already been recorded."
        )


    timestamp = (
        datetime.now()
        .isoformat()
    )


    question_prompt = f"""
USER QUESTION:

{question}

Analyze the complete raw event log provided in the base
interaction and answer the question using only that event log.
"""


    request_start = (
        time.perf_counter()
    )


    try:

        interaction = (
            client.interactions.create(

                model=GEMINI_MODEL,

                input=question_prompt,

                previous_interaction_id=(
                    BASE_INTERACTION_ID
                ),

                system_instruction=(
                    SYSTEM_INSTRUCTION
                ),

                generation_config={
                    "temperature":
                        TEMPERATURE
                },

                store=True
            )
        )


        request_end = (
            time.perf_counter()
        )


        latency = (
            request_end
            -
            request_start
        )


        response_text = (
            interaction.output_text
        )


        input_tokens = None
        cached_tokens = None
        non_cached_input_tokens = None
        output_tokens = None
        total_tokens = None


        try:

            usage = (
                interaction.usage
            )


            input_tokens = getattr(
                usage,
                "total_input_tokens",
                None
            )


            cached_tokens = getattr(
                usage,
                "total_cached_tokens",
                None
            )


            output_tokens = getattr(
                usage,
                "total_output_tokens",
                None
            )


            total_tokens = getattr(
                usage,
                "total_tokens",
                None
            )


            if (
                input_tokens is not None
                and
                cached_tokens is not None
            ):

                non_cached_input_tokens = (
                    input_tokens
                    -
                    cached_tokens
                )


        except Exception:

            pass


        record = {

            "question_id":
                question_id,

            "timestamp":
                timestamp,

            "question":
                question,

            "architecture":
                "LLM-only",

            "response_type":
                "llm_direct",

            "context_strategy":
                "server-side base interaction",

            "base_interaction_id":
                BASE_INTERACTION_ID,

            "interaction_id":
                interaction.id,

            "response":
                response_text,

            "latency_seconds":
                round(
                    latency,
                    6
                ),

            "input_tokens":
                input_tokens,

            "cached_tokens":
                cached_tokens,

            "non_cached_input_tokens":
                non_cached_input_tokens,

            "output_tokens":
                output_tokens,

            "total_tokens":
                total_tokens,

            "rate_limit_error":
                False,

            "error":
                None
        }


        experiment_records.append(
            record
        )


        save_records()


        return {
            "success":
                True,

            "record":
                record
        }


    except Exception as error:

        request_end = (
            time.perf_counter()
        )


        latency = (
            request_end
            -
            request_start
        )


        error_message = (
            str(error)
        )


        normalized_error = (
            error_message.lower()
        )


        is_rate_limit = (
            "429" in error_message
            or
            "resource_exhausted"
            in normalized_error
            or
            "too_many_requests"
            in normalized_error
            or
            "rate limit"
            in normalized_error
        )


        record = {

            "question_id":
                question_id,

            "timestamp":
                timestamp,

            "question":
                question,

            "architecture":
                "LLM-only",

            "response_type":
                "llm_error",

            "context_strategy":
                "server-side base interaction",

            "base_interaction_id":
                BASE_INTERACTION_ID,

            "interaction_id":
                None,

            "response":
                None,

            "latency_seconds":
                round(
                    latency,
                    6
                ),

            "input_tokens":
                None,

            "cached_tokens":
                None,

            "non_cached_input_tokens":
                None,

            "output_tokens":
                None,

            "total_tokens":
                None,

            "rate_limit_error":
                is_rate_limit,

            "error":
                error_message
        }


        experiment_records.append(
            record
        )


        save_records()


        return {
            "success":
                False,

            "record":
                record
        }


# =============================================================================
# Experimental user interface
# =============================================================================

last_llm_use = 0


question_id_box = (
    widgets.Text(

        placeholder="Q01",

        description="Question ID:",

        layout=widgets.Layout(
            width="40%"
        )
    )
)


question_box = (
    widgets.Textarea(

        placeholder=(
            "Enter the exact experimental question..."
        ),

        description="Question:",

        layout=widgets.Layout(
            width="100%",
            height="110px"
        )
    )
)


ask_button = (
    widgets.Button(
        description="Ask",
        button_style="primary"
    )
)


output = (
    widgets.Output()
)


def on_ask_click(_):

    global last_llm_use


    question_id = (
        question_id_box
        .value
        .strip()
        .upper()
    )


    question = (
        question_box
        .value
        .strip()
    )


    with output:

        output.clear_output()


    if question_id == "":

        with output:

            print(
                "Enter the Question ID "
                "(for example, Q01)."
            )

        return


    if question == "":

        with output:

            print(
                "Enter the experimental question."
            )

        return


    elapsed = (
        time.time()
        -
        last_llm_use
    )


    if (
        last_llm_use != 0
        and
        elapsed < MINIMUM_LLM_INTERVAL
    ):

        remaining = int(
            MINIMUM_LLM_INTERVAL
            -
            elapsed
        )


        with output:

            print(
                "Please wait approximately "
                f"{remaining} seconds before "
                "the next LLM request."
            )

        return


    ask_button.disabled = True

    ask_button.description = (
        "Processing..."
    )


    try:

        result = (
            ask_about_log(
                question=question,
                question_id=question_id
            )
        )


        record = (
            result["record"]
        )


        # A request attempt occurred.
        last_llm_use = (
            time.time()
        )


        with output:

            if result["success"]:

                display(
                    Markdown(
                        f"### {record['question_id']}"
                    )
                )


                display(
                    Markdown(
                        f"**Question:**  \n"
                        f"{record['question']}"
                    )
                )


                display(
                    Markdown(
                        f"**Answer:**  \n"
                        f"{record['response']}"
                    )
                )


                print(
                    "\n--- EXPERIMENTAL METADATA ---"
                )

                print(
                    "Latency:",
                    record[
                        "latency_seconds"
                    ],
                    "seconds"
                )

                print(
                    "Input tokens:",
                    record[
                        "input_tokens"
                    ]
                )

                print(
                    "Cached tokens:",
                    record[
                        "cached_tokens"
                    ]
                )

                print(
                    "Non-cached input tokens:",
                    record[
                        "non_cached_input_tokens"
                    ]
                )

                print(
                    "Output tokens:",
                    record[
                        "output_tokens"
                    ]
                )

                print(
                    "Total tokens:",
                    record[
                        "total_tokens"
                    ]
                )

                print(
                    "\nResult saved to:",
                    RESULTS_FILE
                )


                question_id_box.value = ""

                question_box.value = ""


            else:

                if record[
                    "rate_limit_error"
                ]:

                    print(
                        "API rate limit reached."
                    )

                    print(
                        "The failed attempt was "
                        "recorded in the results file."
                    )

                    print(
                        "Do not immediately retry "
                        "the request."
                    )

                else:

                    print(
                        "The request failed."
                    )

                    print(
                        record["error"]
                    )


    finally:

        ask_button.disabled = False

        ask_button.description = (
            "Ask"
        )


ask_button.on_click(
    on_ask_click
)


# =============================================================================
# Ready summary
# =============================================================================

print("\n==========================================")
print("LLM-ONLY BASELINE READY")
print("==========================================")

print(
    "Experiment status:",
    "FROZEN OFFICIAL VERSION"
)

print(
    "Input integrity:",
    "PASSED"
)

print(
    "XLSX rows:",
    original_rows
)

print(
    "CSV rows:",
    csv_rows
)

print(
    "Columns:",
    csv_columns
)

print(
    "Raw CSV characters:",
    raw_csv_characters
)

print(
    "Raw event-log tokens:",
    raw_log_tokens
)

print(
    "Raw CSV SHA-256:",
    raw_csv_sha256
)

print(
    "Model:",
    GEMINI_MODEL
)

print(
    "Temperature:",
    TEMPERATURE
)

print(
    "Base interaction:",
    BASE_INTERACTION_ID
)

print(
    "No deterministic Process Mining "
    "analytics are available to the model."
)

print(
    "Every question independently references "
    "the same complete raw-event-log base."
)

print(
    "Question-answer pairs are not chained."
)

print(
    "Configuration file:",
    CONFIGURATION_FILE
)

print(
    "Results file:",
    RESULTS_FILE
)

print("==========================================\n")


display(
    Markdown(
        "## Official LLM-only Experiment"
    )
)


display(
    question_id_box,
    question_box,
    ask_button,
    output
)
